In [1]:
# Risk-coverage schematic: AURC = E-AURC + AURC*. Drawn in the thesis style (code/thesis_style.py),
# same type scale, line weights and palette as C2_evaluation / attribute_explainability.
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sys.path.insert(0, os.path.abspath("../code"))
import importlib, thesis_style
importlib.reload(thesis_style)     # pick up edits to the style file without a kernel restart
from thesis_style import *
apply()

OUT_DIR = "."
ERR_RATE, N, SEED = 0.20, 20_000, 0
TAU_COV = 0.70                     # coverage of the marked operating point

# Synthetic kappa: failures score lower on average, so they are rejected first.
rng   = np.random.default_rng(SEED)
fail  = rng.random(N) < ERR_RATE
kappa = rng.normal(np.where(fail, 0.0, 1.6), 1.0)
order = np.argsort(-kappa)                         # most confident first
cov   = np.arange(1, N + 1) / N
risk  = np.cumsum(fail[order]) / np.arange(1, N + 1)
r0    = risk[-1]                                   # full-coverage risk = failure rate

# Perfect kappa*: every failure ranked last -> zero risk until coverage 1 - r0.
risk_or = np.clip((cov - (1 - r0)) / cov, 0, None)

i_tau = int(TAU_COV * N) - 1
c_tau, r_tau = cov[i_tau], risk[i_tau]

COL_K    = FAMILY["attr"]           # the example confidence score
COL_OR   = INK                      # perfect kappa*
FILL_OR  = SHADE                    # AURC*: unavoidable, neutral
HATCH_K  = RAMP["attr"][1]          # E-AURC: kappa's own excess, same hue as kappa
plt.rcParams["hatch.linewidth"] = LW["hair"]
plt.rcParams["hatch.color"]     = HATCH_K

fig, ax = plt.subplots(figsize=(120 * MM, 82 * MM))
ax.grid(axis="y", zorder=0); ax.set_axisbelow(True)

# Areas first, curves on top.
ax.fill_between(cov, 0, risk_or, color=FILL_OR, lw=0, zorder=1)
ax.fill_between(cov, risk_or, risk, facecolor="none", edgecolor=HATCH_K, hatch="/////", lw=0, zorder=1)

msp_line(ax, r0)                                     # random kappa: risk constant at r0
ax.plot(cov, risk_or, color=COL_OR, lw=LW["ref"], zorder=3)
ax.plot(cov, risk, color=COL_K, lw=LW["line"], zorder=4)

# Operating point at tau.
ax.plot([c_tau, c_tau], [0, r_tau], color=MUTED, ls=":", lw=LW["hair"], zorder=2)
ax.plot([0, c_tau], [r_tau, r_tau], color=MUTED, ls=":", lw=LW["hair"], zorder=2)
ax.plot(c_tau, r_tau, "o", ms=4, mfc="white", mec=INK, mew=LW["axis"] + 0.2, zorder=5)
ax.plot(1, r0, "o", ms=4, color=INK, zorder=5)

# Annotations: base size for statements, small/muted for labels on lines.
ax.text(0.02, r0 + 0.006, r"random $\kappa$ (no ranking): risk stays constant",
        fontsize=FS["small"], color=MUTED, va="bottom")
ax.annotate(r"full coverage: Risk($\tau_0$) = classifier failure rate",
            xy=(1, r0), xytext=(0.98, 0.235), ha="right", va="center", fontsize=FS["small"],
            arrowprops=dict(arrowstyle="-", color=INK, lw=LW["hair"], shrinkA=2, shrinkB=3))
ax.text(c_tau - 0.02, r_tau + 0.024,
        rf"operating point at threshold $\tau$" "\n"
        rf"accepts {c_tau:.0%} of predictions, {r_tau:.1%} of them failures",
        fontsize=FS["small"], ha="right", va="center", linespacing=1.4)
ax.text(0.42, 0.012, "E-AURC", fontsize=FS["base"], fontweight="bold", color=COL_K, ha="center",
        va="center", bbox=dict(fc="white", ec="none", pad=1.2), zorder=6)
ax.text(0.91, 0.012, "AURC*", fontsize=FS["base"], fontweight="bold", color=INK, ha="center",
        va="center", zorder=6)

ax.set_xlim(0, 1); ax.set_ylim(0, 0.25)
ax.set_xticks(np.linspace(0, 1, 6)); ax.set_xticklabels(["0", "0.2", "0.4", "0.6", "0.8", "1"])
ax.set_yticks(np.arange(0, 0.26, 0.05)); ax.set_yticklabels(["0"] + [f"{v:.2f}" for v in np.arange(0.05, 0.26, 0.05)])
ax.set_xlabel("Coverage (fraction of predictions accepted)")
ax.set_ylabel("Risk (failure rate among accepted)")

handles = [line_handle(COL_K, r"Example $\kappa$", marker=None),
           Line2D([], [], color=COL_OR, lw=LW["ref"], label=r"Perfect $\kappa^*$"),
           msp_handle(r"Random $\kappa$"),
           Patch(facecolor=FILL_OR, lw=0, label="AURC*: unavoidable,\nset by the classifier"),
           Patch(facecolor="none", edgecolor=HATCH_K, hatch="/////", lw=0,
                 label="E-AURC: excess due to\nthe ranking of " r"$\kappa$")]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.01, 0.76), labelspacing=0.6)

for ext in ("pdf", "png"):
    fig.savefig(os.path.join(OUT_DIR, f"rc_curve.{ext}"))
plt.show()
print(f"failure rate {r0:.3f} | operating point cov {c_tau:.2f}, risk {r_tau:.3f}")


failure rate 0.198 | operating point cov 0.70, risk 0.068
